# Add Storm temperature and IVT from ERA5

For each basin, picks ERA5 cells on the low-elevation (< 1000 m) side of the mountains, just upwind of the basin, and describes each storm by conditions there:
- **Temperature:** 600–900 hPa mean temperature over the storm's days, in the cells where specific humidity is in the top 20% for that storm.
- **IVT:** mean IVT over the storm's days in the cells where IVT is in the top 20% for that storm.

Two sets of storms are processed:
1. **SWE-period storms** (1984–2021): grouped storms from the storm notebook. Temperature is added.
2. **GridMET-period storms** (full GridMET/ERA5 record): runs of consecutive GridMET storm days in each basin. Temperature and IVT are added.

# Inputs
| File | From | Used for |
|------|------|----------|
| `Data/Cleaned/ERA5/era5_complete_ivt.nc4` | ERA5 notebook | Daily `t`, `q` (600–900 hPa) and `ivt` on the 0.25° ERA5 grid |
| `Data/DEM/coarsened.nc4` | DEM notebook | 15-arcsec elevation for the whole study area; coarsened to 0.25° to find cells below 1000 m |
| `Data/Cleaned/<basin>_grouped_storm_ds_1.nc4` | Storm notebook (Step 3) | SWE-period storm groups: `Start_Date`, `End_Date`, `SWE`, `Duration`, `WY` |
| `Data/Cleaned/GridMET/<basin>_storm_days.nc4` | GridMET notebook | Days with a GridMET storm anywhere in the basin |
| `config.py` / `utils.py` | | `data_folder`, `Basins`, `basin_names_list`, `shapes` / `calculate_water_year` |

## Outputs (in `Data/Cleaned/`)
**`<basin>_grouped_storm_ds_final.nc4`**: SWE-period storms, one per `Group` (some variables also by `ElevationBin` 2–4)
| Variable | Dims | Description |
|----------|------|-------------|
| `Start_Date` | Group | First day of the storm (any bin) |
| `SWE` | Group, ElevationBin | Area-weighted mean SWE gain during the storm [mm] |
| `Duration` | Group, ElevationBin | Storm length in each bin [days] |
| `WY` | Group | Water year |
| `Temperature` | Group | Storm temperature [K] |

**`<basin>_storms_long.nc4`**: GridMET-period storms, one per `Group`
| Variable | Dims | Description |
|----------|------|-------------|
| `Start_Date` | Group | First storm day |
| `Duration` | Group | Number of consecutive storm days |
| `WY` | Group | Water year |
| `Temperature` | Group | Storm temperature [K] |
| `IVT` | Group | Storm IVT [kg m⁻¹ s⁻¹] |

An optional cell also checks how sensitive storm temperature is to the 80th-percentile cutoff. It prints results only and saves nothing.


In [ ]:
# %% Imports and settings
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import ListedColormap
from tqdm import tqdm

from config import data_folder, Basins, basin_names_list, shapes
from utils import calculate_water_year

cleaned = os.path.join(data_folder, 'Data/Cleaned')
QUANTILE = 0.8      # storm cells = top 20% of q (or IVT) within the mask
LOW_ELEV = 1000     # m
N_EAST_CELLS = 5    # easternmost low-elevation cells kept per latitude row

In [ ]:
# %% ERA5 and DEM on the ERA5 grid
# Small domain, so load once: makes the per-storm loops much faster
era5 = (xr.open_dataset(os.path.join(cleaned, 'ERA5/era5_complete_ivt.nc4'))[['t', 'q', 'ivt']]
        .sortby('latitude').sortby('longitude')
        .load())

# coarsened.nc4 is the 15-arcsec DEM from the DEM notebook; 60 cells = 0.25° (ERA5 grid spacing)
dem = (xr.open_dataset(os.path.join(data_folder, 'Data/DEM/coarsened.nc4'))['Elevation']
       .rename({'Latitude': 'latitude', 'Longitude': 'longitude'}))
dem_coarse = dem.coarsen(latitude=60, longitude=60, boundary='pad').mean().interp_like(era5)

In [ ]:
# %% Masks and per-storm metrics
def low_elev_mask(basin_name):
    """ERA5 cells below LOW_ELEV, from the domain's west edge to the basin's east edge,
    within 0.5° of the basin's latitude range."""
    minx, miny, maxx, maxy = shapes.loc[shapes['name'] == basin_name].total_bounds
    dem_basin = dem_coarse.sel(latitude=slice(miny - 0.5, maxy + 0.5), longitude=slice(None, maxx))
    return ((dem_basin < LOW_ELEV).astype(int)
            .interp(latitude=era5.latitude, longitude=era5.longitude, method='nearest')
            .fillna(0).astype(bool))


def keep_eastern_cells(mask, n=N_EAST_CELLS):
    """Keep only the n easternmost True cells in each latitude row (the valley edge below the mountains)."""
    rev = slice(None, None, -1)
    n_at_or_east = mask.isel(longitude=rev).cumsum('longitude').isel(longitude=rev)
    return mask & (n_at_or_east <= n)


def storm_windows(storms):
    """(start, end) dates for each storm. Uses End_Date when the file has it;
    otherwise end = start + Duration − 1 (exact for runs of consecutive days)."""
    start = storms['Start_Date'].values
    if 'End_Date' in storms:
        return list(zip(start, storms['End_Date'].values))
    return [(s, s + np.timedelta64(int(d) - 1, 'D')) for s, d in zip(start, storms['Duration'].values)]


def storm_temperature(start, end, mask, quantile=QUANTILE):
    """Level-mean temperature over the storm, in mask cells where level-mean q is above its `quantile`."""
    days = slice(start, end)
    t = era5['t'].sel(valid_time=days).mean('pressure_level')
    q = era5['q'].sel(valid_time=days).mean('pressure_level').where(mask)
    return t.where(q > q.quantile(quantile, skipna=True).item()).mean().item()


def storm_ivt(start, end, mask, quantile=QUANTILE):
    """Mean IVT over the storm, in mask cells where IVT is above its `quantile`."""
    ivt = era5['ivt'].sel(valid_time=slice(start, end)).where(mask)
    return ivt.where(ivt > ivt.quantile(quantile, skipna=True).item()).mean().item()

In [ ]:
# %% Build masks (eastern cells only, used for every storm metric); one figure to check them
masks = {}
fig, axes = plt.subplots(1, len(Basins), figsize=(2.6 * len(Basins), 4), sharey=True, constrained_layout=True)

for ax, basin, name in zip(axes, Basins, basin_names_list):
    full = low_elev_mask(name)
    masks[basin] = keep_eastern_cells(full)

    dem_coarse.plot(ax=ax, cmap='Greys', add_colorbar=False)
    full.where(full).plot(ax=ax, cmap=ListedColormap(['#fdd0a2']), add_colorbar=False)
    masks[basin].where(masks[basin]).plot(ax=ax, cmap=ListedColormap(['#d94801']), add_colorbar=False)
    shapes[shapes['name'] == name].boundary.plot(ax=ax, color='k', linewidth=0.6)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle(f'Low-elevation cells (< {LOW_ELEV} m, light) and the {N_EAST_CELLS} easternmost per row (dark, used)')
plt.show()

In [ ]:
# %% Optional check: does the q quantile matter? (diagnostic only; saves nothing)
def quantile_sensitivity(storms, mask, quantiles=np.arange(0.70, 0.96, 0.05)):
    """Share of variance in storm temperature explained by storm vs. q quantile (two-way, no replication)."""
    temps = pd.DataFrame([[storm_temperature(s, e, mask, qq) for qq in quantiles]
                          for s, e in tqdm(storm_windows(storms))],
                         columns=quantiles)  # rows = storms, columns = quantiles
    grand = np.nanmean(temps.values)
    storm_mean, q_mean = temps.mean(axis=1), temps.mean(axis=0)
    resid = temps.sub(storm_mean, axis=0).sub(q_mean, axis=1) + grand
    ss = {
        'Storm': ((storm_mean - grand) ** 2).sum() * temps.shape[1],
        'q quantile': ((q_mean - grand) ** 2).sum() * temps.shape[0],
        'Residual': np.nansum(resid.values ** 2),
    }
    total = sum(ss.values())
    print(', '.join(f'{k}: {v / total:.2f}' for k, v in ss.items()))


for basin in Basins:
    print(basin)
    storms = xr.open_dataset(os.path.join(cleaned, f'{basin}_grouped_storm_ds_1.nc4'))
    quantile_sensitivity(storms, masks[basin])

In [ ]:
# %% 1) SWE-period storms: add Temperature
for basin in Basins:
    grouped = xr.open_dataset(os.path.join(cleaned, f'{basin}_grouped_storm_ds_1.nc4'))
    storms = grouped[['Start_Date', 'SWE', 'Duration', 'WY']]
    storms['Temperature'] = ('Group', [storm_temperature(s, e, masks[basin])
                                       for s, e in storm_windows(grouped)])  # windows use End_Date
    storms.to_netcdf(os.path.join(cleaned, f'{basin}_grouped_storm_ds_final.nc4'))

In [ ]:
# %% 2) GridMET-period storms: group consecutive storm days, add Temperature and IVT
def gridmet_storms(basin):
    """One storm per run of consecutive GridMET storm days in the basin."""
    day = xr.open_dataset(os.path.join(cleaned, f'GridMET/{basin}_storm_days.nc4'))['Day'].values
    group = np.cumsum(np.r_[True, np.diff(day) != np.timedelta64(1, 'D')]) - 1
    by_group = pd.Series(day).groupby(group)

    storms = xr.Dataset({'Duration': ('Group', by_group.size().values),
                         'Start_Date': ('Group', by_group.min().values)},
                        coords={'Group': np.arange(group.max() + 1)})
    storms['WY'] = calculate_water_year(storms['Start_Date'])
    return storms


for basin in Basins:
    storms = gridmet_storms(basin)
    windows = storm_windows(storms)
    storms['Temperature'] = ('Group', [storm_temperature(s, e, masks[basin]) for s, e in windows])
    storms['IVT'] = ('Group', [storm_ivt(s, e, masks[basin]) for s, e in windows])
    storms.to_netcdf(os.path.join(cleaned, f'{basin}_storms_long.nc4'))